#### Enigma Main Encryption Block

In [6]:
class Rotor:
    """Represents a single Enigma rotor with its wiring and position"""
    
    def __init__(self, wiring, notch, ring_setting=0, position=0):
        """
        Args:
            wiring: String of 26 letters representing the rotor's internal wiring
            notch: Letter(s) where rotor causes next rotor to step
            ring_setting: Ring setting (0-25, default 0 for 'A')
            position: Initial rotor position (0-25, default 0 for 'A')
        """
        self.wiring = wiring
        self.notch = notch
        self.ring_setting = ring_setting
        self.position = position
    
    def encode_forward(self, char_index):
        """Encode character passing through rotor from right to left"""
        # Adjust for rotor position and ring setting
        shifted = (char_index + self.position - self.ring_setting) % 26
        # Pass through wiring
        encoded = ord(self.wiring[shifted]) - ord('A')
        # Adjust back
        return (encoded - self.position + self.ring_setting) % 26
    
    def encode_backward(self, char_index):
        """Encode character passing through rotor from left to right (after reflector)"""
        # Adjust for rotor position and ring setting
        shifted = (char_index + self.position - self.ring_setting) % 26
        # Find inverse mapping in wiring
        encoded = self.wiring.index(chr(shifted + ord('A')))
        # Adjust back
        return (encoded - self.position + self.ring_setting) % 26
    
    def is_at_notch(self):
        """Check if rotor is at notch position (will cause next rotor to turn)"""
        return chr(self.position + ord('A')) in self.notch
    
    def step(self):
        """Rotate the rotor by one position"""
        self.position = (self.position + 1) % 26


class Reflector:
    """Represents the Enigma reflector"""
    
    def __init__(self, wiring):
        """
        Args:
            wiring: String of 26 letters representing reflector pairs
        """
        self.wiring = wiring
    
    def reflect(self, char_index):
        """Reflect the character back through the rotors"""
        return ord(self.wiring[char_index]) - ord('A')


class EnigmaMachine:
    """Complete Enigma machine with 3 rotors and reflector"""
    
    # Historical rotor wirings (Enigma I)
    ROTOR_I = "EKMFLGDQVZNTOWYHXUSPAIBRCJ"
    ROTOR_II = "AJDKSIRUXBLHWTMCQGZNPYFVOE"
    ROTOR_III = "BDFHJLCPRTXVZNYEIWGAKMUSQO"
    ROTOR_IV = "ESOVPZJAYQUIRHXLNFTGKDCMWB"
    ROTOR_V = "VZBRGITYUPSDNHLXAWMJQOFECK"
    
    # Reflector B (most commonly used)
    REFLECTOR_B = "YRUHQSLDPXNGOKMIEBFZCWVJAT"
    
    # Notch positions for each rotor
    NOTCHES = {
        'I': 'Q',    # Rotor I turns rotor II when moving from Q to R
        'II': 'E',   # Rotor II turns rotor III when moving from E to F
        'III': 'V',  # Rotor III turns nothing (it's the leftmost)
        'IV': 'J',
        'V': 'Z'
    }
    
    def __init__(self, rotor_types=('I', 'II', 'III'), 
                 positions=(0, 0, 0), ring_settings=(0, 24, 0)):
        """
        Initialize Enigma machine
        
        Args:
            rotor_types: Tuple of 3 rotor identifiers (rightmost, middle, leftmost)
            positions: Starting positions for each rotor (0-25 for A-Z)
            ring_settings: Ring settings for each rotor (0-25 for A-Z)
        """
        rotor_wirings = {
            'I': self.ROTOR_I,
            'II': self.ROTOR_II,
            'III': self.ROTOR_III,
            'IV': self.ROTOR_IV,
            'V': self.ROTOR_V
        }
        
        # Create rotors (right, middle, left)
        self.rotors = [
            Rotor(rotor_wirings[rotor_types[0]], self.NOTCHES[rotor_types[0]], 
                  ring_settings[0], positions[0]),
            Rotor(rotor_wirings[rotor_types[1]], self.NOTCHES[rotor_types[1]], 
                  ring_settings[1], positions[1]),
            Rotor(rotor_wirings[rotor_types[2]], self.NOTCHES[rotor_types[2]], 
                  ring_settings[2], positions[2])
        ]
        
        self.reflector = Reflector(self.REFLECTOR_B)
    
    def step_rotors(self):
        """
        Step rotors according to Enigma stepping mechanism
        Implements the double-stepping mechanism
        """
        # Check if middle rotor is at notch (double-stepping)
        if self.rotors[1].is_at_notch():
            self.rotors[1].step()
            self.rotors[2].step()
        # Check if right rotor is at notch
        elif self.rotors[0].is_at_notch():
            self.rotors[1].step()
        
        # Always step the rightmost rotor
        self.rotors[0].step()
    
    def encode_char(self, char):
        """
        Encode a single character through the Enigma machine
        
        Args:
            char: Single uppercase letter A-Z
            
        Returns:
            Encoded uppercase letter A-Z
        """
        if not char.isalpha():
            return char  # Return non-alphabetic characters unchanged
        
        char = char.upper()
        char_index = ord(char) - ord('A')
        
        # Step rotors before encoding (historical behavior)
        self.step_rotors()
        
        # Pass through rotors right to left
        for rotor in self.rotors:
            char_index = rotor.encode_forward(char_index)
        
        # Reflect
        char_index = self.reflector.reflect(char_index)
        
        # Pass back through rotors left to right
        for rotor in reversed(self.rotors):
            char_index = rotor.encode_backward(char_index)
        
        return chr(char_index + ord('A'))
    
    def encode_text(self, text):
        """
        Encode a string of text
        
        Args:
            text: String to encode
            
        Returns:
            Encoded string
        """
        result = []
        for char in text:
            result.append(self.encode_char(char))
        return ''.join(result)
    
    def get_rotor_positions(self):
        """Get current rotor positions as letters"""
        return ''.join(chr(r.position + ord('A')) for r in self.rotors)


# Example usage
if __name__ == "__main__":
    # Create Enigma machine with rotors I, II, III starting at AAA
    enigma = EnigmaMachine(
        rotor_types=('I', 'II', 'III'),
        positions=(5,25, 19),  # AAA
        ring_settings=(0, 0, 0)  # AAA
    )
    
    print("Enigma Machine Emulator")
    print("=" * 50)
    print(f"Initial rotor positions: {enigma.get_rotor_positions()}")
    print()

Enigma Machine Emulator
Initial rotor positions: FZT



#### Part we care about

In [10]:
import os

# Encode a message from file
plaintext_path = r"C:\Users\tapia\Desktop\Python stuff\Classes\Capstone\training data\Discussion1.txt"

# Build output path: same directory, "Encrypted " + original filename
directory, filename = os.path.split(plaintext_path)
output_path = os.path.join(directory, f"Encrypted {filename}")

# Read and normalize plaintext
with open(plaintext_path, 'r') as f:
    plaintext = ''.join(c for c in f.read().upper() if c.isalpha())

print(f"Plaintext length: {len(plaintext)} characters")

# Encrypt
ciphertext = enigma.encode_text(plaintext)
print(f"Encryption complete.")
print(f"Final rotor positions: {enigma.get_rotor_positions()}")
print()

# Save encrypted output
with open(output_path, 'w') as f:
    f.write(ciphertext)

print(f"Encrypted output written to: {output_path}")
print()

# Sanity check: plaintext preview
print("Plaintext preview:")
print(f"  First 50: {plaintext[:50]}")
print(f"  Last  50: {plaintext[-50:]}")
print()

# Sanity check: ciphertext preview
print("Ciphertext preview:")
print(f"  First 50: {ciphertext[:50]}")
print(f"  Last  50: {ciphertext[-50:]}")
print()


Plaintext length: 2762 characters
Encryption complete.
Final rotor positions: DZK

Encrypted output written to: C:\Users\tapia\Desktop\Python stuff\Classes\Capstone\training data\Encrypted Discussion1.txt

Plaintext preview:
  First 50: FORTHEDISCUSSIONTOPICICHOSETHECONCERNSONACCURACYAN
  Last  50: SAREANDHOWTOFIXTHEMWHICHIBELIEVEISAGREATWAYTOUSEIT

Ciphertext preview:
  First 50: GNUXVYRJTDJMTFGSLFGXBVIZAFUSZRQSGKINWZNWEJMLVWLNNQ
  Last  50: EOPGRSGRIFHGEEFOMVQPRRUBYIJBNRXMEQZANMVDIQRZRWGPGE

